# TP Jour 1 — Choix du Sujet Kaggle, Stockage Distribué HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et votre jeu de données public sur **Kaggle**.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner un dataset Kaggle adapté au calcul distribué (~50 à 500 Mo).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts sur HDFS** et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** et inspecter votre premier job sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** `NYC Yellow Taxi Trip Records (Analyse de mobilité)`
- **Identifiant Kaggle (`auteur/nom-du-dataset`) :** `elemento/nyc-yellow-taxi-trip-data` 
- **Problématique métier principale :** `Prédire ou analyser les tendances de courses de taxi en fonction des horaires et identifier les zones à forte demande.`
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :** `tpep_pickup_datetime, trip_distance, passenger_count, total_amount`

## 1. Téléchargement de votre Dataset Kaggle dans `/workspace/data_raw`

Renseignez dans la cellule suivante l'identifiant Kaggle de votre dataset (ex. `"auteur/mon-dataset"`), ou déposez directement vos fichiers dans le dossier `data_raw/` à la racine du projet.
> **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez l'identifiant Kaggle de votre sujet (format "auteur/nom-du-dataset")
KAGGLE_DATASET = "elemento/nyc-yellow-taxi-trip-data"

local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

if not local_files:
    assert KAGGLE_DATASET.strip(), (
        "TODO : Renseignez `KAGGLE_DATASET = 'auteur/nom-du-dataset'` (ou déposez vos fichiers dans data_raw/)."
    )
    import kagglehub
    print(f"Téléchargement de {KAGGLE_DATASET} depuis Kaggle Hub...")
    cache_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    for src_file in sorted(cache_dir.rglob("*")):
        if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
            shutil.copy2(src_file, DATA_RAW / src_file.name)
    local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, "Aucun fichier de données détecté dans /workspace/data_raw/."

print(f"Inventaire local dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {f.name:<45} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local dans /workspace/data_raw (4 fichier(s)) :
 - yellow_tripdata_2015-01.csv                   1893.96 Mo  (~119 bloc(s) HDFS de 16 Mo)
 - yellow_tripdata_2016-01.csv                   1629.52 Mo  (~102 bloc(s) HDFS de 16 Mo)
 - yellow_tripdata_2016-02.csv                   1700.93 Mo  (~107 bloc(s) HDFS de 16 Mo)
 - yellow_tripdata_2016-03.csv                   1825.97 Mo  (~115 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession

# Initialisation de la SparkSession configurée pour HDFS / Delta Lake
spark = (
    SparkSession.builder
    .appName("J1_HDFS_Ingestion_Bronze")
    .getOrCreate()
)

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# Lecture des 3 paramètres HDFS demandés par le TP
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

:: loading settings :: url = jar:file:/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0ee1bbf1-5675-4b23-bd76-3a1861b47fff;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta

Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{file_path.name}")
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/yellow_tripdata_2015-01.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/yellow_tripdata_2016-01.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/yellow_tripdata_2016-02.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/yellow_tripdata_2016-03.csv


In [4]:
# Inspection physique du découpage en blocs de 16 Mo et identification du plus gros fichier pour le crash-test
statuses = sorted(fs.listStatus(raw_hdfs_dir), key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {path.getName()} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : yellow_tripdata_2015-01.csv (1893.96 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #11 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #12 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #13 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #14 (16.00 Mo) -> DataNodes : ['

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
Pour mon fichier le plus volumineux (yellow_tripdata_2015-01.csv avec $\approx 1893.96$ Mo), si on divise par 16 Mo, on obtient environ $118,37$ blocs. Dans votre sortie précédente, on voyait exactement 119 blocs (du Bloc #0 au Bloc #118, le dernier faisant 5,96 Mo). Chaque bloc de 16 Mo correspond à une tranche physique du fichier.

2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
Non. Contrairement à une idée reçue, HDFS n'alloue pas la taille maximale du bloc sur le disque pour un petit fichier : il ne consomme que l'espace réel qu'il occupe (soit ~3 Mo).

3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?
Cela offre un compromis idéal pour le développement : on assure une tolérance aux pannes (chaque bloc est dupliqué sur 2 nœuds différents) tout en économisant de l'espace disque par rapport à une réplication complète à 3.
---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

Le comptage s'effectue sans aucune erreur malgré l'arrêt du datanode2, et le nombre de partitions RDD correspond exactement au nombre total de blocs HDFS qui composent le fichier

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [6]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# TODO : Lire `largest_hdfs_path` avec PySpark (selon son format : spark.read.csv / parquet / json)
df_check = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(largest_hdfs_path)
)

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/yellow_tripdata_2015-01.csv


Nombre total de lignes lues       : 12,748,986
Nombre de partitions Spark créées : 15


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts en tables Delta Lake Bronze
Dans une architecture **Médaillon**, la couche **Bronze** conserve les données brutes fidèles à la source tout en les stockant au format transactionnel **Delta Lake** enrichi de métadonnées de traçabilité :
- `_ingested_at` : `F.current_timestamp()`
- `_source_file` : `F.input_file_name()`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de votre sujet dans `/lakehouse/bronze/<nom_table>`.

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialisation de la SparkSession avec le support Delta Lake
spark = (
    SparkSession.builder
    .appName("Lakehouse Bronze Ingestion")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

print("SparkSession initialisée avec succès !")

# 2. Chemin source dans HDFS (selon votre configuration précédente)
source_path = "hdfs://namenode:9000/lakehouse/bronze/raw/*.csv"

# 3. Lecture des fichiers CSV bruts avec inférence du schéma et en-têtes
df_raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

# 4. Normalisation des noms de colonnes en snake_case (suppression des espaces et caractères spéciaux)
new_columns = [
    c.strip().lower().replace(" ", "_").replace("(", "").replace(")", "")
    for c in df_raw.columns
]
df_renamed = df_raw.toDF(*new_columns)

# 5. Ajout des métadonnées de traçabilité requises
df_bronze = (
    df_renamed
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name())
)

# 6. Écriture au format Delta Lake dans la couche Bronze
delta_bronze_path = "hdfs://namenode:9000/lakehouse/bronze/nyc_taxi_delta"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .save(delta_bronze_path)
)

print(f"Couche Bronze Delta Lake écrite avec succès vers : {delta_bronze_path}")

26/11/17 11:12:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
26/11/17 11:12:46 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://namenode:9000/lakehouse/bronze/raw/*.csv.
java.io.FileNotFoundException: File does not exist: hdfs://namenode:9000/lakehouse/bronze/raw/*.csv
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.anal

SparkSession initialisée avec succès !


26/11/17 11:13:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Couche Bronze Delta Lake écrite avec succès vers : hdfs://namenode:9000/lakehouse/bronze/nyc_taxi_delta


In [8]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['nyc_taxi_delta']
